# Synthetic Kannada caller turns (#7, #12)

Runs `training/generate_data.py`: labels are sampled from `data/generation/scenarios.json`, Qwen3-14B writes each caller sentence in spoken Kannada, and only sentences that pass the consistency checks are kept.

**How to run:** Run → **Run all**. Nothing needs editing.
1. Setup and secrets.
2. Pilot: 40 sentences. Skipped if it already ran in this session.
3. Samples: each sentence next to what it was asked to say. Paste these into the chat for checking.
4. Full run: 2,500 sentences, uploaded to the private dataset repo. **Off until `RUN_FULL = True`.**

**Before running**
- Kaggle: Settings → Accelerator **GPU T4 x2**, Internet **on**; Add-ons → Secrets → `HF_TOKEN` (a write token), ticked for this notebook.
- Colab: Runtime → **T4 GPU**; Secrets (key icon) → `HF_TOKEN`, notebook access on.

Why Qwen3-14B:
- Qwen2.5-7B wrote mostly non-Kannada word strings.
- Qwen3-8B got about a third of sentences wrong in meaning.
- Sarvam-M (24B) was a 47 GB download and too slow on T4 GPUs to finish a batch.
- Qwen3-14B passed 39 of 40 checks on the first try.

In [ ]:
import os
import subprocess

ROOT = '/kaggle/working' if os.path.isdir('/kaggle/working') else '/content'
REPO_DIR = f'{ROOT}/dhwani-desk'
BRANCH = 'main'  # or a feature branch, to try changes before they merge
if not os.path.isdir(REPO_DIR):
    url = 'https://github.com/Pranav-0710/dhwani-desk.git'
    subprocess.run(['git', 'clone', '--depth', '1', '-b', BRANCH, url, REPO_DIR], check=True)
os.chdir(f'{REPO_DIR}/training')
# transformers 4.56+ reads the `dtype` argument; older versions ignore it and load in fp32.
!pip install -q 'transformers>=4.56' accelerate bitsandbytes huggingface_hub

In [ ]:
import torch

try:
    from kaggle_secrets import UserSecretsClient

    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
except ImportError:
    from google.colab import userdata

    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

DATASET_REPO = 'Dhwanidesk/dhwani-desk-data'  # private; see STRUCTURE.md
# Qwen3-14B is ~30 GB in fp16, more than 2 x 15 GB of T4 memory, so it always loads in 4-bit.
MODEL = 'Qwen/Qwen3-14B'
print(torch.cuda.device_count(), 'GPU(s)')
assert torch.cuda.is_available(), 'no GPU: set the accelerator to GPU T4 x2 and run again'

## Pilot: 40 sentences

In [ ]:
PILOT_OUT = 'out/pilot-qwen3-14b'
if os.path.exists(f'{PILOT_OUT}/report.json'):
    print('Pilot already done in this session; showing its samples below.')
else:
    !python generate_data.py --n 40 --model {MODEL} --out {PILOT_OUT} --four-bit

In [ ]:
import json

with open(f'{PILOT_OUT}/report.json', encoding='utf-8') as f:
    report = json.load(f)
print(f"kept {report['kept']}/{report['requested']}, "
      f"first-try pass {report['first_attempt_pass_rate']:.0%}, {report['rejection_reasons']}")
print()
with open(f'{PILOT_OUT}/train.jsonl', encoding='utf-8') as f:
    rows = [json.loads(line) for line in f]
for row in rows[:20]:
    print(f"{row['id']} | asked: {row['meta']['scenario']}")
    print(f"          wrote: {row['text']}")

## Full run: 2,500 sentences and upload

Only after the pilot samples are checked. Set `RUN_FULL = True` and run this cell. It takes a few hours (Kaggle allows 12 per session). It writes `train.jsonl`, `rejected.jsonl`, `report.json` and a dataset card, then uploads them to `DATASET_REPO`. Tagging the version and the 10% hand check follow in #12.

In [ ]:
RUN_FULL = False

if RUN_FULL:
    !python generate_data.py --n 2500 --model {MODEL} --out out/train --four-bit --push-repo {DATASET_REPO}
else:
    print('Full run is off. Set RUN_FULL = True in this cell to start it.')